Winter 2026 COG260H1S LEC5101

Can S Mekik

# Lab 6: Mini Project 2: Bandits in `pyClarion` — SOLUTION

The questions on this lab are worth a total of 12 points. There are no bonus questions.

A **bandit task** is a very simple kind of reinforcement learning task. It is 
named after slot machines, so-called 'one-armed bandits' (because they rob 
gambling addicts).

Unlike the casino setting, where the house always wins, the bandit task involves 
making discrete choices which may probabilistically yield rewards or punishments 
and learning which choices yield the best rewards. So, you see, it is a very 
nice and simple setting to flex some cognitive modeling skills.

In the classic bandit setup, rewards are stochastic (i.e. random) and the 
agent's policy is typically deterministic. However, in our setting, we will do 
things a little bit differently, mainly to simplify the implementation. In 
particular, we will have a deterministic reward pattern but a stochastic policy.

You can think of this task as `pyClarion`'s 'Hello World!' for reinforcement 
learning. Actually, what we will have the model do is literally respond to 
greetings. So we will try to get the model to learn to say 'hi' in response to 
'hi' and 'bye' in response to 'bye'. The model will receive reward for correct 
behavior, and punishment for incorrect behavior. 

Your task will be to implement such a model mostly from scratch, with some 
scaffolding of course. More specifically, you will be implementing a very simple 
agent that makes decisions using a Q-learning neural network with a single layer 
and an identity transfer function.

For this, we will make use of `pyClarion`'s support for **temporal difference 
learning**, which is a broad class of reinforcement learning methods that 
include Q-learning, and **reverse mode automatic differentiation**, which is a 
fancy way of saying 'automatic backpropagation'.

## Imports

Here are the `pyClarion` classes you will need to use for this assignment.

As always, please make sure to update your installation before you attempt this exercise.

In [5]:
import random
import math
import pandas as pd
from typing import TypedDict
from datetime import timedelta
from plotnine import ggplot, aes, geom_smooth, geom_line

from pyClarion.knowledge import Root, BusFamily, Buses, Bus, AtomFamily, Atoms, Atom
from pyClarion import Key, Event, Priority, Agent, Input, Layer
from pyClarion.components.base import ForwardUpdate
from pyClarion.components.learning import TDLearning
from pyClarion.components.optimizers import Adam

## Keyspace

Here is the model keyspace. 

In [6]:
class Message(Atoms):
    """
    Data symbols for message types. 

    The symbol 'nil' represents no message.
    """
    nil: Atom
    hi: Atom
    bye: Atom 


class Reward(Atoms):
    """
    Data symbols for reward inputs.
    
    For this model, there is only one type of reward input, so we call it 
    'main'.
    """
    main: Atom


class ModelData(AtomFamily):
    """
    Model's data symbol family.
    
    Consists only of messages and rewards.
    """
    message: Message
    reward: Reward


class MainBuses(Buses):
    """
    Main buses for model.
    
    Since we are dealing only with external state (e.g. no goals, etc.), we 
    only need a main bus for wm.
    """
    wm: Bus


class ModelLayout(BusFamily):
    """
    Family of model buses.

    Since there is no complex control processes (e.g. buffers, stacks, etc.), we 
    only need a set of main buses to hold state data.
    """
    main: MainBuses
    

class ModelKeyspace(Root):
    b: ModelLayout
    d: ModelData
    p: AtomFamily

## Question 1: Agent Construction

Below is the completed agent based on the provided instructions and docstrings. 

**Implementation note:** The structure follows the patterns established in the introduction tutorial:
- Components are constructed within a `with self:` block so that they are auto-registered with the agent's system.
- Processes are chained with `>>` to form a forward computation graph (e.g. `ipt >> layer >> out`).
- The `Layer` carries weight/bias parameter sites; these have to be added to the optimizer's scope explicitly so that gradients flow through them during the update step.
- Following hint 3, rewards live in *dimension-value* form, so the reward input is paired with a bus on the dimension side and the reward atom on the value side.

In [7]:
class TrainStats(TypedDict):
    """Statistical data about the model's behavior in training."""
    trial: list[int]
    stimulus: list[str]
    cost: list[float]
    qmax: list[float]


class Model(Agent):
    root: ModelKeyspace
    ipt: Input
    layer: Layer
    out: TDLearning
    optimizer: Adam

    # ---------------------------------------------------------------------
    # Q1a (1 pt): Build the agent's processes and wire them together.
    # ---------------------------------------------------------------------
    def __init__(self, 
        name: str, 
        root: ModelKeyspace, 
        *, 
        lr: float = 1e-2, 
        gamma = .7
    ) -> None:
        super().__init__(name, root)
        self.root = root

        # Convenient handles for keyspace symbols.
        main    = root.b.main          # main bus sort
        message = root.d.message       # message atoms (nil, hi, bye)
        reward  = root.d.reward        # reward atoms (main)

        # Components instantiated inside this block are auto-registered
        # with the agent's discrete-event system.
        with self:
            # The input process receives external messages. It carries a
            # single dimension-value domain: (main.wm, message.*). The
            # reward signal is delivered separately, through self.out's own
            # reward site (see self.out.send in train), so it does NOT need
            # its own input channel here.
            self.ipt = Input(
                f"{name}.ipt", 
                (main, message),   # message channel
            )

            # The Q-network is a single Layer with an identity transfer
            # function. l=2 keeps two activation snapshots (current and
            # previous) so Q-learning can compute its target term using
            # the previous step (hint 1).
            #
            # Inputs to the layer come from the message channel of self.ipt
            # and outputs are also indexed over messages (Q-values for each
            # possible response: nil/hi/bye).
            self.layer = self.ipt >> Layer(
                f"{name}.layer",
                (main, message),   # input domain
                (main, message),   # output domain
                l=2,
            )

            # Temporal-difference learning component. TDLearning is a Choice
            # process, so it needs a parameter family (p) and a status
            # family (s) in addition to its decision domain (d) and reward
            # domain (r). We reuse root.p for both p and s; the component
            # creates distinct parameter/status sorts within it. It reads
            # the layer's current and one-step-lagged output together with
            # the reward signal to compute the TD error.
            self.out = self.layer >> TDLearning(
                f"{name}.out",
                root.p,            # p: parameter family
                root.p,            # s: status family
                (main, message),   # d: decision (action) domain
                (main, reward),    # r: reward domain  (hint 3)
                gamma=gamma,
            )

            # Adam optimizer. Like all Parametric processes it takes a
            # parameter family (root.p). The Layer has two parameter sites
            # -- weights and bias -- that have to be explicitly added to the
            # optimizer's scope so that updates actually touch them
            # (hint 2).
            self.optimizer = Adam(f"{name}.optimizer", root.p, lr=lr)
            self.optimizer.add(self.layer.weights, self.layer.bias)

    # ---------------------------------------------------------------------
    # Q1b (1 pt): On every forward update of the layer, trigger a
    # downstream response-selection step. The Agent base class lets us
    # do that by intercepting events as they fire.
    # ---------------------------------------------------------------------
    def resolve(self, event: Event) -> None:
        """Trigger response selection on a forward update to self.layer."""
        # `event` is an Event, which carries a list of Updates. We watch for
        # events that write a ForwardUpdate to the layer's output site
        # (self.layer.main): when one fires, the new Q-values are fresh and
        # a response can be sampled. Choice.trigger() schedules the
        # stochastic selection.
        if self.layer.main in event.index(ForwardUpdate):
            self.system.schedule(self.out.trigger())

    # ---------------------------------------------------------------------
    # Q1c (2 pts): Initialize layer parameters from N(0, sd).
    # We return a single Event holding TWO ForwardUpdate's -- one for the
    # weight site and one for the bias site -- so they are written
    # atomically.
    # ---------------------------------------------------------------------
    def initialize_weights(self, 
        sd: float, 
        dt: timedelta = timedelta(), 
        priority=Priority.LEARNING
    ) -> Event:
        """
        Initialize weights and biases for self.layer.
        
        Sets each weight and bias to a random value sampled from N(0, sd).
        """
        # Build dictionaries mapping each weight/bias key to a fresh
        # Gaussian sample. Iterating over an index gives us its keys
        # (hint 2).
        weight_init = {
            k: random.gauss(0.0, sd)
            for k in self.layer.weights.index
        }
        bias_init = {
            k: random.gauss(0.0, sd)
            for k in self.layer.bias.index
        }

        # Event signature is Event(source, [updates], time, priority). We
        # pass both ForwardUpdate's in a single list so they are applied
        # together, scheduled at the requested offset (dt) and priority.
        return Event(
            self.initialize_weights,
            [ForwardUpdate(self.layer.weights, weight_init),
             ForwardUpdate(self.layer.bias,    bias_init)],
            dt,
            priority,
        )

    # ---------------------------------------------------------------------
    # Q1d (1 pt): Reward signal for a given (stimulus, choice) pair.
    #   choice == nil      -> 0.0   (the model has not committed)
    #   choice == stimulus -> +1.0  (correct response)
    #   choice != stimulus -> -1.0  (incorrect response)
    # ---------------------------------------------------------------------
    def compute_reward(self, stimulus: Key) -> dict[Key, float]:
        """
        Generates reward for the given stimulus based on current choice output.
        """
        message = self.root.d.message
        reward  = self.root.d.reward
        main    = self.root.b.main

        # The choice the model just made lives in the choice process's main
        # output site (a one-hot numdict over the (wm, message) domain). Its
        # argmax key is the chosen action -- a full dimension-value key of
        # the form (main, message):(wm, <atom>), so we must compare it to
        # likewise-constructed Keys (atom inverted into the (wm, *) domain),
        # not to bare atoms.
        choice = self.out.main[0].argmax()

        if choice == ~(main.wm * message.nil):
            r = 0.0
        elif choice == ~(main.wm * stimulus):
            r = +1.0
        else:
            r = -1.0

        # Hint 3: rewards are returned as a dimension-value-keyed dict. The
        # key is built with `*` (which combines paths) and materialized into
        # a Key with `~`; `**` would instead build a weighted Chunk.
        return {~(main.wm * reward.main): r}

    # ---------------------------------------------------------------------
    # Q1e (2 pts): One training cycle = (i) select a response, (ii)
    # compute & deliver the reward so the TD error is backpropagated, (iii)
    # update parameters.
    # ---------------------------------------------------------------------
    def train(self, stimuli: list[Key], n: int = 500) -> TrainStats:
        """
        Train self on a random sequence of drawn from stimuli.
        """
        main = self.root.b.main

        stats: TrainStats = {
            "trial":    [],
            "stimulus": [],
            "cost":     [],
            "qmax":     [],
        }

        for trial in range(n):
            # ------- (i) Select a response based on a given stimulus -------
            stim = random.choice(stimuli)
            # Send the stimulus into the input process as a one-hot
            # activation (+1.0 on the chosen message dim). This drives the
            # forward pass, which (via resolve) triggers selection and the
            # TDLearning update.
            self.system.schedule(
                self.ipt.send(+ main.wm ** stim)
            )
            self.run_all()

            # ------- (ii) Compute and deliver the reward ------------------
            # The reward is delivered to the TDLearning component's own
            # reward site via self.out.send. TDLearning computes
            #   delta = r + gamma * max_a' Q(s', a') - Q(s, a)
            # and backpropagates the error through the layer.
            reward_dict = self.compute_reward(stim)
            self.system.schedule(self.out.send(reward_dict))

            # ------- (iii) Update model parameters ------------------------
            # Adam consumes the gradients accumulated at the layer's weight
            # and bias sites and applies an update.
            self.system.schedule(self.optimizer.update())
            self.run_all()

            # ------- Record statistics for this trial ---------------------
            qvals = self.layer.main[0]
            cost  = float(self.out.cost[0].valmax())  # 0.5 * delta^2
            qmax  = float(qvals.valmax())

            stats["trial"].append(trial)
            stats["stimulus"].append(str(stim))
            stats["cost"].append(cost)
            stats["qmax"].append(qmax)

        return stats


## Question 2: Simulation

### Question 2a (0.5 pts)

Initialize the model and report its initial weights and biases. 

In [8]:
# Q2a -------------------------------------------------------------------
# Build the keyspace and the agent, then sample initial weights from
# N(0, 0.1) and print them.

random.seed(0)  # reproducibility

root  = ModelKeyspace()
model = Model("agent", root, lr=1e-2, gamma=0.3)

# Schedule the weight-initialization event and run it through.
model.system.schedule(model.initialize_weights(sd=0.1))
for ev in model.run():
    pass

print("Initial weights:")
print(model.layer.weights[0])
print()
print("Initial biases:")
print(model.layer.bias[0])

ValueError: Key 'p:agent.out:gamma' not a member

### Question 2b (0.5 pts)

Then, train the model on 999 samples. Use learning weight `1e-2` and discount value 0.3.

In [ ]:
# Q2b -------------------------------------------------------------------
# Train on a sequence of 999 random {hi, bye} stimuli.

message = root.d.message
stimuli = [message.hi, message.bye]

stats = model.train(stimuli, n=999)

df = pd.DataFrame(stats)
df.head()

### Question 2c (0.5 pts)

Plot the training cost as a function of trials. Use both `geom_line` with 
an alpha value of .3, and `geom_smooth`.

What do you observe?

In [ ]:
# Q2c -------------------------------------------------------------------
# Plot per-trial cost (raw, with low alpha) plus a smoothed trend.

(
    ggplot(df, aes(x="trial", y="cost"))
    + geom_line(alpha=0.3)
    + geom_smooth()
)

**Observation.** The raw cost curve is noisy because the policy is stochastic — 
even a well-trained agent occasionally samples a wrong response, producing a 
non-zero TD error. Despite the noise, the smoothed trend declines sharply over 
the first ~100–200 trials and then plateaus close to (but above) zero. This is 
the signature of successful Q-learning: as the weights move toward values that 
make `Q(hi, hi) > Q(hi, bye), Q(hi, nil)` and similarly for `bye`, the average 
TD error shrinks. The residual non-zero floor is expected — it reflects the 
occasional stochastic mis-pick rather than a systematic learning failure.

### Question 2d (0.5 pts)

Report the weights and biases obtained after training. Interpret how the learned parameters control the model's behavior.

In [ ]:
# Q2d -------------------------------------------------------------------
# Inspect the learned parameters.

print("Learned weights:")
print(model.layer.weights[0])
print()
print("Learned biases:")
print(model.layer.bias[0])

**Interpretation.** The Q-network here is `Q(s, a) = W·s + b` with an identity 
transfer function. With messages `nil/hi/bye` on both the input and output 
sides, `W` is effectively a 3×3 matrix indexed by (input message, output 
action) and `b` is a 3-vector indexed by output action.

For the learning to succeed, the diagonal weights `W[hi, hi]` and `W[bye, bye]` 
should grow large and positive — i.e. when the agent sees `hi`, the unit 
encoding the action `hi` should receive the strongest excitation, and likewise 
for `bye`. The off-diagonal *cross* weights `W[hi, bye]` and `W[bye, hi]` 
should be driven negative or near zero, since selecting the wrong message earns 
−1 reward. Weights tied to the `nil` row (input is `nil`, which never happens 
during training) and the `nil` column (action is `nil`, which always earns 0) 
should drift relatively little — they only get very weak gradient signal.

The biases pick up baseline preferences: `b[nil]` will be roughly 0 because the 
`nil` action always receives 0 reward and provides no learning signal of its 
own; `b[hi]` and `b[bye]` may end up slightly negative because the *average* 
reward for blindly emitting `hi` (or `bye`) across both stimuli is 
`(+1 − 1) / 2 = 0`, partially cancelled by the diagonal weights when the right 
stimulus is present. In short, the learned matrix becomes approximately 
diagonal, which is exactly the linear separator needed to map each input 
directly to its matching action.

## Question 3: Integrative Questions

These are some conceptual questions about the neural network lecture that tie it back to other topics in the course.

### Question 3a (1 pt)

A neural network with multiple layers can learn the XOR problem and other non-separable and complicated categories. However, this only works if the nodes have non-linear transfer functions. Why?

#### Q3a Answer

Stacking layers with linear transfer functions doesn't actually buy any 
expressive power, because **the composition of linear functions is itself 
linear**. Concretely, if layer 1 computes $h = W_1 x + b_1$ and layer 2 
computes $y = W_2 h + b_2$, then

$$
y \;=\; W_2 (W_1 x + b_1) + b_2 \;=\; (W_2 W_1)\, x + (W_2 b_1 + b_2),
$$

which is just *one* linear (affine) transformation $W' x + b'$. So no matter 
how many linear layers you stack, the network can only carve up the input 
space with a single hyperplane — i.e. it can only realize **linearly 
separable** decision boundaries.

XOR is the canonical counter-example: the four points $(0,0), (0,1), (1,0), 
(1,1)$ with their XOR labels cannot be split by any single straight line, so a 
purely linear network of any depth provably cannot learn it.

Inserting non-linear activations (sigmoid, tanh, ReLU, …) breaks this 
collapse. The hidden layer can then *bend* the input space — for example, 
carving out two half-planes whose intersection isolates the XOR-positive 
points — and the next layer can combine those bent regions into the desired 
non-linear decision boundary. This is the mechanism behind the Universal 
Approximation Theorem: a feed-forward network with at least one hidden layer 
of *non-linear* units can approximate any continuous function on a compact 
domain to arbitrary precision.

### Question 3b (1 pt)

What conceptual connection, if any, is there between backpropagation and the secant formula we discussed in Week 4?

#### Q3b Answer

Both are doing the same conceptual thing — *measuring how much one quantity 
changes in response to a change in another* — but at different levels of 
fidelity.

The **secant formula** approximates a derivative with a finite difference:

$$
f'(x) \;\approx\; \frac{f(x + h) - f(x)}{h}.
$$

It estimates the slope of $f$ at $x$ by drawing a straight line (a *secant*) 
between two evaluations of $f$ separated by a small step $h$. As $h \to 0$ the 
secant slope converges to the true tangent slope.

**Backpropagation** computes the *exact* analytic derivative of the loss with 
respect to every weight, by mechanically applying the chain rule layer by 
layer from the output back to the input. No finite step is required; the 
gradients fall out symbolically because we know each layer's transfer 
function in closed form.

So the connection is conceptual: **both are tools for asking the same 
question — "if I nudge this parameter, how does the output change?"** The 
secant formula answers it numerically, with one black-box evaluation per 
parameter; backpropagation answers it analytically, propagating partial 
derivatives through a known computational graph. Indeed, you can sanity-check 
a backprop implementation by comparing its gradients against finite 
differences computed with the secant formula — a standard practice known as 
*gradient checking*. Backprop is just the limit of the secant idea made 
exact, structural, and shared across many parameters at once.

There is also a second, more goal-directed sense in which the two connect.
A derivative is not just a description of local change; it is the quantity
that tells us *which way to move* to make a function larger or smaller. The
secant formula gives a numerical estimate of that slope, and in one dimension
a root-finding or optimization routine can use it to step toward where the
derivative vanishes — i.e. toward a minimum (or maximum) of the function.
Backpropagation supplies the exact, high-dimensional analogue: the gradient it
computes is precisely the direction of steepest increase of the loss, so
gradient descent steps *against* it to drive the loss downhill toward a
minimum. In other words, both the secant slope and the backpropagated gradient
are ultimately in service of the same optimization goal — locating a point
where the error is (locally) smallest — with backprop scaling that idea up to
every weight in the network at once.


### Question 3c (1 pt)

Give a probabilistic interpretation of the inputs and outputs of an artificial neuron with logistic activation function.

#### Q3c Answer

A logistic (sigmoid) neuron computes

$$
y \;=\; \sigma(z) \;=\; \frac{1}{1 + e^{-z}}, \qquad z \;=\; \sum_i w_i x_i + b.
$$

Because $\sigma(\cdot)$ squashes $z \in (-\infty, \infty)$ into $(0, 1)$, the 
output is naturally read as a **probability**.

**Output interpretation.** $y$ is the (Bernoulli) probability that some binary 
event $C = 1$ holds given the inputs:

$$
P(C = 1 \mid x) \;=\; \sigma(z),\qquad P(C = 0 \mid x) \;=\; 1 - \sigma(z).
$$

**Input / pre-activation interpretation.** The pre-activation $z$ is exactly 
the **log-odds** (the *logit*) of that probability:

$$
z \;=\; \log \frac{P(C = 1 \mid x)}{P(C = 0 \mid x)}.
$$

Each input $x_i$ contributes additively to that log-odds with weight $w_i$, 
and the bias $b$ sets a baseline log-odds when all inputs are zero. Equivalently, 
the neuron is implementing **logistic regression**: under the assumption that 
$P(x \mid C)$ comes from a member of the exponential family with shared 
covariance (e.g. Gaussians with shared $\Sigma$, or independent Bernoullis as 
in naive Bayes), Bayes' rule yields a posterior $P(C \mid x)$ that is exactly a 
logistic function of an affine combination of the inputs. So a single sigmoid 
unit can be read as the posterior probability of class 1 under such generative 
models, with the weights mediating the evidence each input feature provides 
for that class.